# Chebyshev Energy Cutoff on a Narrow Spectral Window

This notebook is about a regime where the Chebyshev energy cutoff is actually doing meaningful work instead of being numerically invisible.

We use the same disordered transverse-field Ising model (TFIM) in two stages:

1. a small chain where we can diagonalize the Hamiltonian exactly and see which spectral weight really lies inside a chosen low-energy window,
2. a larger chain where exact diagonalization is no longer practical, so we compare a low-budget MPS calculation against a more expensive reference.

The key point is that the cutoff is most instructive when we intentionally focus on a **narrow target window** `|E'| <= window < 1` after the usual Chebyshev rescaling. In easy full-window calculations with `window = 1`, the difference can be tiny. Here we deliberately make the problem harder so the truncation has something real to clean up.


## Model and helper functions

We work with the open-chain disordered TFIM

$$H = -J \sum_{j=1}^{L-1} S^z_j S^z_{j+1} - g \sum_{j=1}^L S^x_j - \sum_{j=1}^L h_j S^z_j.$$

The helper code below does four jobs:

- build the MPO and, for small systems, the dense Hamiltonian,
- find a ground state with DMRG,
- create the local probe state $|\phiangle = S^z_{j_0}|gsangle$,
- run Chebyshev moments with and without `energy_cutoff=true` and reconstruct spectra on a shared frequency grid.


In [ ]:
using LinearAlgebra
using Random
using Plots
using ITensors
using ITensorMPS
using MPSToolkit
using EDKit

default(; linewidth=2, legend=:topright)

function disordered_tfim_mpo(sites; J::Real=1.0, g::Real=0.9, hz)
    nsites = length(sites)
    opsum = OpSum()
    for j in 1:(nsites - 1)
        opsum += -J, "Sz", j, "Sz", j + 1
    end
    for j in 1:nsites
        opsum += -g, "Sx", j
        opsum += -hz[j], "Sz", j
    end
    return MPO(opsum, sites)
end

function kron_all(ops)
    out = ops[1]
    for op in ops[2:end]
        out = kron(out, op)
    end
    return out
end

function dense_disordered_tfim(nsites; J::Real=1.0, g::Real=0.9, hz)
    sx = ComplexF64[0 1; 1 0] / 2
    sz = ComplexF64[1 0; 0 -1] / 2
    id = Matrix{ComplexF64}(I, 2, 2)
    H = zeros(ComplexF64, 2^nsites, 2^nsites)

    for j in 1:(nsites - 1)
        ops = [id for _ in 1:nsites]
        ops[j] = sz
        ops[j + 1] = sz
        H .+= -J .* kron_all(ops)
    end

    for j in 1:nsites
        opsx = [id for _ in 1:nsites]
        opsx[j] = sx
        H .+= -g .* kron_all(opsx)

        opsz = [id for _ in 1:nsites]
        opsz[j] = sz
        H .+= -hz[j] .* kron_all(opsz)
    end

    return Hermitian(H)
end

function dmrg_ground_state(H::MPO, sites; rng=MersenneTwister(0))
    psi0 = random_mps(rng, sites; linkdims=4)
    energy, state = dmrg(
        H,
        psi0;
        nsweeps=6,
        maxdim=[16, 32, 64, 96, 128, 192],
        cutoff=1e-10,
        outputlevel=0,
    )
    return energy, state
end

function local_probe_state(ground_state::MPS, sites, center_site::Integer)
    probe = apply(op("Sz", sites[center_site]), ground_state; maxdim=256, cutoff=1e-12)
    normalize!(probe)
    return probe
end

function chebyshev_run(h_rescaled::MPO, probe::MPS, halfwidth::Real;
    order::Integer,
    maxdim::Integer,
    grid_points::Integer=500,
    energy_cutoff::Bool=false,
    window::Real=1.0,
    energy_cutoff_sweeps::Integer=4,
    krylovdim::Integer=16,
)
    moments = chebyshev_moments(
        h_rescaled,
        probe;
        order=order,
        maxdim=maxdim,
        cutoff=1e-10,
        energy_cutoff=energy_cutoff,
        energy_cutoff_sweeps=energy_cutoff_sweeps,
        krylovdim=krylovdim,
        window=window,
        energy_cutoff_tol=1e-12,
    )
    ω_grid = collect(range(-halfwidth, halfwidth; length=grid_points))
    spectrum = spectral_function(moments; center=0.0, halfwidth=halfwidth, kernel=:jackson)
    return (; moments, ω_grid, ρ_grid=spectrum(ω_grid))
end

function low_window_mask(ω_grid, halfwidth, window)
    abs.(ω_grid) .<= window * halfwidth
end


## Small exact-reference case

We first choose a chain small enough for exact diagonalization. The probe is still local, but now we can measure exactly how much spectral weight lies inside our target low-energy window.

This section is the trust-building part of the notebook:

- the exact sticks tell us what should survive inside the low-energy window,
- the no-cutoff Chebyshev run still carries information from the whole spectral bandwidth,
- the cutoff run actively projects the recursion back into the target window after each step.


In [ ]:
small_seed = 4
small_rng = MersenneTwister(small_seed)

small_nsites = 6
small_order = 120
small_budget_maxdim = 8
small_window = 0.45
small_disorder_strength = 1.2
small_center = small_nsites ÷ 2

small_hz = small_disorder_strength .* randn(small_rng, small_nsites)
small_sites = siteinds("S=1/2", small_nsites)
small_H = disordered_tfim_mpo(small_sites; hz=small_hz)
small_dense_H = dense_disordered_tfim(small_nsites; hz=small_hz)

small_ground_energy_mps, small_ground_state = dmrg_ground_state(small_H, small_sites; rng=small_rng)
small_probe = local_probe_state(small_ground_state, small_sites, small_center)

small_exact_energies, small_exact_vectors = eigen(small_dense_H)
small_ground_energy_exact = small_exact_energies[1]
small_excitation_energies = small_exact_energies .- small_ground_energy_exact
small_probe_vector = ComplexF64.(EDKit.mps2vec(small_probe))
small_stick_weights = abs2.(small_exact_vectors' * small_probe_vector)

small_halfwidth = maximum(abs, small_excitation_energies) + 0.05
small_h_rescaled = (small_H - small_ground_energy_exact * MPO(small_sites, "Id")) / small_halfwidth
small_in_window = abs.(small_excitation_energies ./ small_halfwidth) .<= small_window

small_plain = chebyshev_run(small_h_rescaled, small_probe, small_halfwidth;
    order=small_order,
    maxdim=small_budget_maxdim,
)
small_cut = chebyshev_run(small_h_rescaled, small_probe, small_halfwidth;
    order=small_order,
    maxdim=small_budget_maxdim,
    energy_cutoff=true,
    window=small_window,
)
small_ref = chebyshev_run(small_h_rescaled, small_probe, small_halfwidth;
    order=small_order,
    maxdim=64,
    energy_cutoff=true,
    window=small_window,
    krylovdim=20,
)

println("small exact in-window spectral weight  = ", sum(small_stick_weights[small_in_window]))
println("small exact out-of-window spectral weight = ", sum(small_stick_weights[.!small_in_window]))
println("small max |plain - cutoff-ref| moments = ", maximum(abs.(small_plain.moments .- small_ref.moments)))
println("small max |cutoff - cutoff-ref| moments = ", maximum(abs.(small_cut.moments .- small_ref.moments)))


### Small-system spectrum comparison

The vertical black sticks are the exact finite-size spectral weights. We shade the target low-energy window and zoom in on that region.

What to look for:

- the exact sticks show that almost all of the spectral weight is already in the low-energy window for this probe,
- the cutoff run is therefore allowed to be selective without throwing away much physical weight,
- the no-cutoff run still carries the influence of the full spectrum and can drift more from the filtered reference under the same `maxdim` budget.


In [ ]:
small_plot = plot(
    xlabel="ω",
    ylabel="A(ω)",
    title="Small exact-reference comparison",
    xlim=(-small_window * small_halfwidth, small_window * small_halfwidth),
)

plot!(small_plot, small_plain.ω_grid, small_plain.ρ_grid; label="no cutoff (budget)", color=:steelblue)
plot!(small_plot, small_cut.ω_grid, small_cut.ρ_grid; label="cutoff (budget)", color=:darkorange)
plot!(small_plot, small_ref.ω_grid, small_ref.ρ_grid; label="cutoff (higher maxdim)", color=:seagreen, linestyle=:dash)

for (ω, weight, keep) in zip(small_excitation_energies, small_stick_weights, small_in_window)
    keep || continue
    plot!(small_plot, [ω, ω], [0.0, weight]; color=:black, alpha=0.45, label=false)
end

vspan!(small_plot, [-small_window * small_halfwidth, small_window * small_halfwidth]; color=:gray90, alpha=0.15, label="target window")
small_plot


### Small-system moments

A moment plot is often the quickest way to see whether two recursion strategies are drifting apart before the final spectrum makes that obvious.


In [ ]:
moment_index_small = 0:(small_order - 1)
small_moment_plot = plot(
    moment_index_small,
    abs.(small_plain.moments);
    yscale=:log10,
    xlabel="moment index n",
    ylabel="|μ_n|",
    title="Small-system moment comparison",
    label="no cutoff (budget)",
    color=:steelblue,
)
plot!(small_moment_plot, moment_index_small, abs.(small_cut.moments); label="cutoff (budget)", color=:darkorange)
plot!(small_moment_plot, moment_index_small, abs.(small_ref.moments); label="cutoff (higher maxdim)", color=:seagreen, linestyle=:dash)
small_moment_plot


## Larger MPS-only comparison

Now we keep the same model family and the same local probe idea, but move to a larger chain where exact diagonalization is no longer our comparison tool.

Instead, we compare three calculations:

- a low-budget run without energy cutoff,
- a low-budget run with energy cutoff,
- a more expensive cutoff run that serves as an internal reference.

This is the regime where the cutoff is meant to be part of a stabilization strategy: we are targeting a narrow low-energy window with a deliberately modest bond-dimension budget.


In [ ]:
large_seed = 11
large_rng = MersenneTwister(large_seed)

large_nsites = 18
large_order = 260
large_budget_maxdim = 6
large_reference_maxdim = 48
large_window = 0.45
large_disorder_strength = 1.8
large_center = large_nsites ÷ 2

large_hz = large_disorder_strength .* randn(large_rng, large_nsites)
large_sites = siteinds("S=1/2", large_nsites)
large_H = disordered_tfim_mpo(large_sites; hz=large_hz)

large_ground_energy, large_ground_state = dmrg_ground_state(large_H, large_sites; rng=large_rng)
large_probe = local_probe_state(large_ground_state, large_sites, large_center)

large_halfwidth = (large_nsites - 1) / 4 + large_nsites * 0.9 / 2 + sum(abs, large_hz) / 2 + 0.05
large_h_rescaled = (large_H - large_ground_energy * MPO(large_sites, "Id")) / large_halfwidth

large_plain = chebyshev_run(large_h_rescaled, large_probe, large_halfwidth;
    order=large_order,
    maxdim=large_budget_maxdim,
)
large_cut = chebyshev_run(large_h_rescaled, large_probe, large_halfwidth;
    order=large_order,
    maxdim=large_budget_maxdim,
    energy_cutoff=true,
    window=large_window,
)
large_ref = chebyshev_run(large_h_rescaled, large_probe, large_halfwidth;
    order=large_order,
    maxdim=large_reference_maxdim,
    energy_cutoff=true,
    window=large_window,
    krylovdim=20,
)

large_mask = low_window_mask(large_plain.ω_grid, large_halfwidth, large_window)
println("large low-window max |no cutoff - reference| = ", maximum(abs.(large_plain.ρ_grid[large_mask] .- large_ref.ρ_grid[large_mask])))
println("large low-window max |cutoff - reference|    = ", maximum(abs.(large_cut.ρ_grid[large_mask] .- large_ref.ρ_grid[large_mask])))
println("large max |no cutoff - cutoff| in window      = ", maximum(abs.(large_plain.ρ_grid[large_mask] .- large_cut.ρ_grid[large_mask])))


### Large-system spectrum comparison

This plot focuses only on the target low-energy region. The qualitative lesson matters more than any one disorder realization:

- under a narrow target window and a small `maxdim`, the budgeted curves are no longer identical,
- the cutoff run is the one that is explicitly trying to keep the recursion inside the desired low-energy sector,
- a higher-budget cutoff run gives a useful internal reference once exact diagonalization is out of reach.

For some realizations the improvement is dramatic, for others it is modest. The notebook is meant to show *where* the cutoff enters the workflow and what kind of regime makes it relevant.


In [ ]:
large_plot = plot(
    xlabel="ω",
    ylabel="A(ω)",
    title="Large-chain low-energy comparison",
    xlim=(-large_window * large_halfwidth, large_window * large_halfwidth),
)
plot!(large_plot, large_plain.ω_grid, large_plain.ρ_grid; label="no cutoff (budget)", color=:steelblue)
plot!(large_plot, large_cut.ω_grid, large_cut.ρ_grid; label="cutoff (budget)", color=:darkorange)
plot!(large_plot, large_ref.ω_grid, large_ref.ρ_grid; label="cutoff (higher maxdim)", color=:seagreen, linestyle=:dash)
vspan!(large_plot, [-large_window * large_halfwidth, large_window * large_halfwidth]; color=:gray90, alpha=0.15, label="target window")
large_plot


### Large-system moments

The moments make it easier to see how much the two low-budget recursion strategies diverge over long order.


In [ ]:
moment_index_large = 0:(large_order - 1)
large_moment_plot = plot(
    moment_index_large,
    abs.(large_plain.moments);
    yscale=:log10,
    xlabel="moment index n",
    ylabel="|μ_n|",
    title="Large-system moment comparison",
    label="no cutoff (budget)",
    color=:steelblue,
)
plot!(large_moment_plot, moment_index_large, abs.(large_cut.moments); label="cutoff (budget)", color=:darkorange)
plot!(large_moment_plot, moment_index_large, abs.(large_ref.moments); label="cutoff (higher maxdim)", color=:seagreen, linestyle=:dash)
large_moment_plot


## Takeaways

This example is intentionally not a claim that the cutoff always changes the answer dramatically. The more practical lesson is narrower:

- if you keep the full Chebyshev window and the problem is easy, the cutoff may do very little,
- if you care about a narrower low-energy window and the recursion is compression-limited, the cutoff gives you a principled way to keep filtering the recursion back into that target sector,
- once exact diagonalization is unavailable, comparing a budgeted run against a more expensive cutoff reference is a reasonable way to decide whether the extra truncation machinery is buying you anything.

In other words, the cutoff is most useful when the calculation is **both** spectrally selective and numerically stressed.
